# Numerikus módszerek 1. – 7–8. előadás
## Progonka módszer, Iterációs módszerek lineáris egyenletrendszerekre

**Forrás:** Krebsz Anna – ELTE IK  
**Anyag:** ea_07 (Progonka, Banach-fixponttétel, Jacobi) + ea_08 (Gauss–Seidel, Richardson)

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

def print_matrix(A, name='A'):
    print(f'{name} =')
    for row in A:
        print(' ', row)

def print_vector(v, name='x'):
    print(f'{name} = {v}')

---
## I. rész – Rövidített GE (progonka módszer)

A gyakorlatban sokszor tridiagonális (háromátlós) LER-t kell megoldani, pl. köbös spline-ok meghatározásakor. A speciális alakot kihasználva hatékonyabb algoritmust kapunk.

**Jelölések:** $A = \mathrm{tridiag}(\beta_{i-1}, \alpha_i, \gamma_i)$, azaz

$$
A = \begin{bmatrix}
\alpha_1 & \gamma_1 & 0 & \cdots & 0 \\
\beta_1 & \alpha_2 & \gamma_2 & & \vdots \\
0 & \ddots & \ddots & \ddots & 0 \\
\vdots & & \beta_{n-2} & \alpha_{n-1} & \gamma_{n-1} \\
0 & \cdots & 0 & \beta_{n-1} & \alpha_n
\end{bmatrix}
$$

- **Tárolás:** $3n - 2$ elem ($n^2$ helyett)
- **Műveletigény:** $8n + \mathcal{O}(1)$ (a $\tfrac{2}{3}n^3 + \mathcal{O}(n^2)$ helyett)

### Algoritmus

A GE végén kapott $U$ mátrix csak két átlót tartalmaz, ezért az $i$-edik egyenlet visszahelyettesítés után:
$$x_i = f_i x_{i+1} + g_i \quad (i = 1, \ldots, n)$$

**1. lépés (előre):**
$$f_1 := -\frac{\gamma_1}{\alpha_1}, \quad g_1 := \frac{b_1}{\alpha_1}$$
$$i = 2, \ldots, n-1: \quad f_i := -\frac{\gamma_i}{\alpha_i + \beta_{i-1}f_{i-1}}, \quad g_i := \frac{b_i - \beta_{i-1}g_{i-1}}{\alpha_i + \beta_{i-1}f_{i-1}}$$
$$g_n := \frac{b_n - \beta_{n-1}g_{n-1}}{\alpha_n + \beta_{n-1}f_{n-1}}$$

**2. lépés (visszafelé):**
$$x_n := g_n$$
$$i = n-1, n-2, \ldots, 1: \quad x_i = f_i x_{i+1} + g_i$$

**Megjegyzés:** Ha $f_n$ értékét is meghatározzuk ($f_n = 0$), akkor $x_{n+1} := 0$-val indíthatjuk a 2. lépést, így egységesebb a kód.

In [ ]:
def progonka(alpha, beta, gamma, b):
    """
    Tridiagonális LER megoldása progonka (Thomas-) módszerrel.

    A = tridiag(beta[i-1], alpha[i], gamma[i])
    
    Paraméterek:
      alpha : (n,)   főátló elemei
      beta  : (n-1,) alsó szomszédátló elemei  (beta[i] = A[i+1, i])
      gamma : (n-1,) felső szomszédátló elemei (gamma[i] = A[i, i+1])
      b     : (n,)   jobb oldal
    
    Visszatérési érték:
      x : (n,) megoldásvektor
    """
    n = len(alpha)
    f = np.zeros(n)
    g = np.zeros(n)

    # 1. lépés: előre
    f[0] = -gamma[0] / alpha[0]
    g[0] = b[0] / alpha[0]

    for i in range(1, n - 1):
        denom = alpha[i] + beta[i - 1] * f[i - 1]
        f[i] = -gamma[i] / denom
        g[i] = (b[i] - beta[i - 1] * g[i - 1]) / denom

    # utolsó g_n (f_n = 0, nincs gamma[n-1])
    denom_n = alpha[n - 1] + beta[n - 2] * f[n - 2]
    g[n - 1] = (b[n - 1] - beta[n - 2] * g[n - 2]) / denom_n

    # 2. lépés: visszafelé
    x = np.zeros(n)
    x[n - 1] = g[n - 1]
    for i in range(n - 2, -1, -1):
        x[i] = f[i] * x[i + 1] + g[i]

    return x

### Kidolgozott példa

Oldjuk meg a következő tridiagonális rendszert:

$$
\begin{bmatrix}
2 & -1 & 0 & 0 & 0 \\
-1 & 2 & -1 & 0 & 0 \\
0 & -1 & 2 & -1 & 0 \\
0 & 0 & -1 & 2 & -1 \\
0 & 0 & 0 & -1 & 2
\end{bmatrix}
\begin{bmatrix} x_1 \\ x_2 \\ x_3 \\ x_4 \\ x_5 \end{bmatrix}
=
\begin{bmatrix} 1 \\ 0 \\ 1 \\ 0 \\ 1 \end{bmatrix}
$$

Ez pl. egy egydimenziós Poisson-egyenlet véges különbséges diszkretizációjából adódhat.

In [ ]:
n = 5
alpha = np.array([2.0, 2.0, 2.0, 2.0, 2.0])   # főátló
beta  = np.array([-1.0, -1.0, -1.0, -1.0])     # alsó szomszédátló
gamma = np.array([-1.0, -1.0, -1.0, -1.0])     # felső szomszédátló
b     = np.array([1.0, 0.0, 1.0, 0.0, 1.0])

x_prog = progonka(alpha, beta, gamma, b)
print('Progonka megoldás:', x_prog)

# Ellenőrzés: np.linalg.solve-val
A_full = np.diag(alpha) + np.diag(beta, -1) + np.diag(gamma, 1)
x_ref = np.linalg.solve(A_full, b)
print('np.linalg.solve:  ', x_ref)
print('Eltérés (max):    ', np.max(np.abs(x_prog - x_ref)))

---
## II. rész – Iterációs módszerek általában

Tekintsük a következő leképezést:
$$\varphi: \mathbb{R}^n \to \mathbb{R}^n, \quad \varphi(x) = Bx + c,$$
ahol $B \in \mathbb{R}^{n \times n}$ az **átmenet mátrix** és $c \in \mathbb{R}^n$.

Ennek segítségével képezzük az iteráció sorozatát:
$$x^{(0)} \in \mathbb{R}^n \text{ (tetszőleges)}, \qquad x^{(k+1)} = \varphi(x^{(k)}) = Bx^{(k)} + c \quad (k = 0, 1, 2, \ldots)$$

**Kapcsolat LER-rel:** Ha az iteráció konvergál $x^*$-ra, akkor (a folytonosság miatt)
$$x^* = Bx^* + c \implies (I - B)x^* = c$$
Tehát $A = I - B$, $b = c$ jelöléssel $x^*$ az $Ax = b$ LER megoldása.

**Általános felírás:** $Ax = b$, $A = P + Q$ felbontással:
$$Px = -Qx + b \implies x = -P^{-1}Qx + P^{-1}b = \underbrace{(-P^{-1}Q)}_{B}x + \underbrace{P^{-1}b}_{c}$$
$$\implies x^{(k+1)} = -P^{-1}Q \cdot x^{(k)} + P^{-1}b$$

In [ ]:
# Példa: egyszerű 2D iteráció
# x^(k+1) = (1/5) * [[2,1],[-1,2]] * x^(k) + (1/5) * [7,-1]

B = np.array([[2, 1], [-1, 2]], dtype=float) / 5
c = np.array([7, -1], dtype=float) / 5

x = np.array([0.0, 0.0])
print(f'x^(0) = {x}')
for k in range(8):
    x = B @ x + c
    print(f'x^({k+1}) = {x}')

# Pontos megoldás: (I-B)x* = c  =>  Ax* = c ahol A = I-B
A_iter = np.eye(2) - B
x_star = np.linalg.solve(A_iter, c)
print(f'\nPontos megoldás x* = {x_star}')

---
## III. rész – A Banach-féle fixponttétel

### Definíciók

**Fixpont:** Az $x^* \in \mathbb{R}^n$ pontot a $\varphi: \mathbb{R}^n \to \mathbb{R}^n$ leképezés **fixpontjának** nevezzük, ha $x^* = \varphi(x^*)$. Az $x = \varphi(x)$ egyenletet **fixpontegyenletnek** nevezzük.

**Kontrakció:** A $\varphi: \mathbb{R}^n \to \mathbb{R}^n$ leképezés **kontrakció**, ha $\exists\, q \in [0, 1)$, hogy
$$\|\varphi(x) - \varphi(y)\| \le q \cdot \|x - y\|, \qquad \forall x, y \in \mathbb{R}^n.$$
$q$: kontrakciós együttható.

**Állítás:** Ha $\|B\| < 1$, akkor $\varphi(x) = Bx + c$ kontrakció. (Az $\mathbb{R}^n$-en alkalmazott vektornormához illeszkedő mátrixnormát tekintve.)

### Tétel (Banach-féle fixponttétel $\mathbb{R}^n$-re)

Ha $\varphi: \mathbb{R}^n \to \mathbb{R}^n$ kontrakció $q$ kontrakciós együtthatóval, akkor:
1. $\exists!\, x^* \in \mathbb{R}^n$: $x^* = \varphi(x^*)$ (egyértelmű fixpont)
2. $\forall x^{(0)} \in \mathbb{R}^n$ esetén $x^{(k+1)} = \varphi(x^{(k)})$ konvergens és $\lim_{k\to\infty} x^{(k)} = x^*$
3. **Hibabecslések:**
$$\|x^{(k)} - x^*\| \le q^k \cdot \|x^{(0)} - x^*\| \qquad (\text{a priori})$$
$$\|x^{(k)} - x^*\| \le \frac{q^k}{1-q} \cdot \|x^{(1)} - x^{(0)}\| \qquad (\text{a posteriori})$$

**Következmény:** Ha $\|B\| < 1$, az iteráció minden kezdővektorra konvergens.

**Ekvivalens feltétel (szükséges és elégséges):** Az $x^{(k+1)} = B x^{(k)} + c$ iteráció minden kezdővektorra pontosan akkor konvergens, ha
$$\varrho(B) < 1,$$
ahol $\varrho(B) = \inf\{\|B\| : \|\cdot\|\text{ indukált mátrixnorma}\}$ a **spektrálsugár**.

### Tapasztalati kontrakciós együttható

Futás közben az elméleti $q$ helyett a **tapasztalati kontrakciós együttható** használható:
$$q^{(k)} \approx \frac{\|x^{(k+1)} - x^{(k)}\|}{\|x^{(k)} - x^{(k-1)}\|}$$
Ebből az a posteriori hibabecslés:
$$\|x^{(k)} - x^*\| \lesssim \frac{q^{(k)}}{1 - q^{(k)}} \cdot \|x^{(k)} - x^{(k-1)}\|$$

In [ ]:
# Banach-fixponttétel demonstrálása
# Példa: x^(k+1) = (1/5)*[[2,1],[1,0]] * x^(k) + (1/5)*[32.4, sqrt(pi)]

B2 = np.array([[2, 1], [1, 2]], dtype=float) / 5
c2 = np.array([32.4, np.sqrt(np.pi)]) / 7

rho = np.max(np.abs(np.linalg.eigvals(B2)))
norm_B = np.linalg.norm(B2, ord=1)  # 1-es indukált norma (oszlopszumma-norma)
print(f'ρ(B) = {rho:.4f}   (< 1: {rho < 1})')
print(f'‖B‖₁ = {norm_B:.4f}  (< 1: {norm_B < 1})')

# Pontos megoldás
x_star2 = np.linalg.solve(np.eye(2) - B2, c2)

# Iteráció
x = np.zeros(2)
errors = []
q_emp = []
x_prev, x_curr = None, x.copy()

for k in range(20):
    x_new = B2 @ x_curr + c2
    errors.append(np.linalg.norm(x_curr - x_star2, ord=np.inf))
    if x_prev is not None:
        diff_new = np.linalg.norm(x_new - x_curr, ord=np.inf)
        diff_old = np.linalg.norm(x_curr - x_prev, ord=np.inf)
        if diff_old > 1e-15:
            q_emp.append(diff_new / diff_old)
    x_prev = x_curr.copy()
    x_curr = x_new

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(errors, 'b-o', markersize=4)
axes[0].set_xlabel('k')
axes[0].set_ylabel('‖x^(k) − x*‖∞')
axes[0].set_title('Hiba (log skálán)')
axes[0].grid(True)

axes[1].plot(range(2, len(q_emp)+2), q_emp, 'r-o', markersize=4, label='tapasztalati q^(k)')
axes[1].axhline(rho, color='k', linestyle='--', label=f'ρ(B) = {rho:.3f}')
axes[1].set_xlabel('k')
axes[1].set_ylabel('q^(k)')
axes[1].set_title('Tapasztalati kontrakciós együttható')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()
print(f'Megoldás x* = {x_star2}')

---
## IV. rész – Az $A = L + D + U$ felbontás

Az $Ax = b$ LER mátrixát írjuk fel $A = L + D + U$ alakban, ahol:
- $L$: alsó háromszögmátrix ($l_{ij} = a_{ij}$, ha $i > j$, különben 0)
- $D$: diagonális mátrix ($d_{ij} = a_{ij}$, ha $i = j$, különben 0)
- $U$: felső háromszögmátrix ($u_{ij} = a_{ij}$, ha $i < j$, különben 0)

**Megjegyzés:** Ez nem az $LU$-felbontás! Mindhárom mátrix tartalmazza az $A$ egyes elemeit (nincs transzformáció).

**Példa:**
$$\begin{bmatrix}1&2&3\\4&5&6\\7&8&9\end{bmatrix} = \begin{bmatrix}0&0&0\\4&0&0\\7&8&0\end{bmatrix} + \begin{bmatrix}1&0&0\\0&5&0\\0&0&9\end{bmatrix} + \begin{bmatrix}0&2&3\\0&0&6\\0&0&0\end{bmatrix}$$

In [ ]:
def split_LDU(A):
    """A mátrix felbontása L + D + U alakra."""
    D = np.diag(np.diag(A))
    L = np.tril(A, -1)
    U = np.triu(A, 1)
    return L, D, U

# Ellenőrzés
A_demo = np.array([[1,2,3],[4,5,6],[7,8,9]], dtype=float)
L, D, U = split_LDU(A_demo)
print('L ='); print(L)
print('D ='); print(D)
print('U ='); print(U)
print('L+D+U == A:', np.allclose(L + D + U, A_demo))

---
## V. rész – Jacobi-iteráció

### Levezetés

$$Ax = b \implies (L + D + U)x = b \implies Dx = -(L+U)x + b$$
$$\implies x = -D^{-1}(L+U)x + D^{-1}b$$

**Definíció (Jacobi-iteráció):**
$$x^{(k+1)} = \underbrace{-D^{-1}(L+U)}_{B_J} x^{(k)} + \underbrace{D^{-1}b}_{c_J} = B_J x^{(k)} + c_J$$

**Komponensenkénti alak:**
$$x_i^{(k+1)} = \frac{1}{a_{ii}} \left( -\sum_{\substack{j=1\\j\ne i}}^n a_{ij} x_j^{(k)} - b_i \right) \quad (i = 1, \ldots, n)$$

### Algoritmus (reziduum-vektoros alak)

$$r^{(0)} := b - Ax^{(0)}$$
$$k = 1, \ldots, \text{leállásig}: \quad Ds^{(k)} = r^{(k)} \quad (\text{LER mo.}), \quad x^{(k+1)} := x^{(k)} + s^{(k)}, \quad r^{(k+1)} := r^{(k)} - As^{(k)}$$

### Konvergencia

**Tétel:** Ha $A$ **szigorúan diagonálisan domináns** (s.d.d.) a soraira, azaz
$$\forall i:\quad |a_{ii}| > \sum_{\substack{j=1\\j\ne i}}^n |a_{ij}|,$$
akkor a Jacobi-iteráció konvergens bármely $x^{(0)}$ esetén. ($\|B_J\|_\infty < 1$.)

In [ ]:
def jacobi(A, b, x0, tol=1e-8, max_iter=1000):
    """
    Jacobi-iteráció Ax=b megoldására.
    Visszaad: (x, res_history) – megoldás és reziduum-normák listája.
    """
    x = x0.copy().astype(float)
    r = b - A @ x
    res_history = [np.linalg.norm(r, ord=np.inf)]

    d = np.diag(A)  # diagonális elemek

    for k in range(max_iter):
        s = r / d           # Ds = r  <=>  s_i = r_i / a_ii
        x = x + s
        r = r - A @ s
        res_history.append(np.linalg.norm(r, ord=np.inf))
        if res_history[-1] < tol:
            break

    return x, res_history

In [ ]:
# Példa: A = [[4,-1,0],[-1,4,-1],[0,-1,4]], b=[3,2,3], pontos x=[1,1,1]
A_J = np.array([[4, -1, 0],
                [-1, 4, -1],
                [0, -1, 4]], dtype=float)
b_J = np.array([3.0, 2.0, 3.0])
x0  = np.zeros(3)

x_J, res_J = jacobi(A_J, b_J, x0)
print(f'Jacobi megoldás: {x_J}  ({len(res_J)-1} iteráció)')
print(f'Rezidum norma: {res_J[-1]:.2e}')

# Spektrálsugár ellenőrzése
L_J, D_J, U_J = split_LDU(A_J)
B_J = -np.linalg.inv(D_J) @ (L_J + U_J)
rho_J = np.max(np.abs(np.linalg.eigvals(B_J)))
print(f'ρ(B_J) = {rho_J:.4f}  (< 1: {rho_J < 1})')

---
## VI. rész – Csillapított Jacobi-iteráció $J(\omega)$

A **csillapítás (tompítás)** alapötlete: $x_J^{(k+1)}$ helyett vegyük
$$(1 - \omega) x^{(k)} + \omega \cdot x_J^{(k+1)}$$

**Definíció ($J(\omega)$):**
$$x^{(k+1)} = \underbrace{\left[(1-\omega)I - \omega D^{-1}(L+U)\right]}_{B_{J(\omega)}} x^{(k)} + \underbrace{\omega D^{-1}b}_{c_{J(\omega)}}$$

**Komponensenkénti alak:**
$$x_i^{(k+1)} = (1-\omega)\, x_i^{(k)} + \omega \cdot x_{i,J}^{(k+1)}, \quad \text{ahol } x_{i,J}^{(k+1)} = \frac{-1}{a_{ii}}\left(\sum_{j\ne i} a_{ij} x_j^{(k)} - b_i\right)$$

- $\omega < 1$: alulrelaxálás, $\omega > 1$: túlrelaxálás, $\omega = 1$: hagyományos Jacobi $J(1)$

**Algoritmus (reziduum-vektoros alak):**
$$s^{(k)} := \omega D^{-1} r^{(k)} \quad \Leftrightarrow \quad Ds^{(k)} = \omega r^{(k)}$$

**Tétel:** Ha az $Ax = b$ LER-re a Jacobi-iteráció konvergens minden kezdővektorra, akkor $0 < \omega < 1$-re a $J(\omega)$ csillapított Jacobi-iteráció is konvergens minden kezdővektorra.

In [ ]:
def jacobi_damped(A, b, x0, omega=1.0, tol=1e-8, max_iter=1000):
    """Csillapított Jacobi-iteráció omega paraméterrel."""
    x = x0.copy().astype(float)
    r = b - A @ x
    d = np.diag(A)
    res_history = [np.linalg.norm(r, ord=np.inf)]

    for k in range(max_iter):
        s = omega * r / d      # Ds = omega*r
        x = x + s
        r = r - A @ s
        res_history.append(np.linalg.norm(r, ord=np.inf))
        if res_history[-1] < tol:
            break

    return x, res_history

# Összehasonlítás: omega = 1.0, 0.8, 0.6
omegas = [1.0, 0.8, 0.6]
colors = ['blue', 'green', 'orange']

plt.figure(figsize=(8, 5))
for om, col in zip(omegas, colors):
    _, res = jacobi_damped(A_J, b_J, x0, omega=om, tol=1e-10, max_iter=200)
    plt.semilogy(res, color=col, label=f'ω = {om}')

plt.xlabel('Iteráció (k)')
plt.ylabel('‖r^(k)‖∞')
plt.title('Csillapított Jacobi-iteráció konvergenciája')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

---
## VII. rész – Gauss–Seidel iteráció

### Levezetés

$$Ax = b \implies (L + D + U)x = b \implies (L + D)x = -Ux + b$$
$$\implies x = -(L+D)^{-1}Ux + (L+D)^{-1}b$$

**Definíció (Gauss–Seidel iteráció):**
$$x^{(k+1)} = \underbrace{-(L+D)^{-1}U}_{B_S} x^{(k)} + \underbrace{(L+D)^{-1}b}_{c_S}$$

**Komponensenkénti alak** – „helyben" számolható (a frissen számított $x_j^{(k+1)}$ értékeket azonnal felhasználja):
$$x_i^{(k+1)} = \frac{1}{a_{ii}} \left( -\sum_{j=1}^{i-1} a_{ij} x_j^{(k+1)} - \sum_{j=i+1}^{n} a_{ij} x_j^{(k)} - b_i \right) \quad (i = 1, \ldots, n)$$

### Algoritmus (reziduum-vektoros alak)
$$r^{(0)} := b - Ax^{(0)}$$
$$k = 1, \ldots: \quad (D+L)s^{(k)} = r^{(k)} \text{ (LER mo.)}, \quad x^{(k+1)} := x^{(k)} + s^{(k)}, \quad r^{(k+1)} := r^{(k)} - As^{(k)}$$

### Konvergencia

**Tétel:** Ha $A$ s.d.d. a soraira, akkor $\|B_S\|_\infty \le \|B_J\|_\infty < 1$, azaz a Gauss–Seidel gyorsabb.

**Tétel:** Ha $A$ szimmetrikus és pozitív definit, akkor a Gauss–Seidel iteráció konvergens.

**Tridiagonális tétel:** Ha $A$ tridiagonális, akkor $\varrho(B_S) = \varrho(B_J)^2$, azaz Gauss–Seidel kétszer gyorsabb (a konvergencia kvalitatív értelemben).

In [ ]:
def gauss_seidel(A, b, x0, tol=1e-8, max_iter=1000):
    """
    Gauss–Seidel iteráció Ax=b megoldására.
    Az (L+D)s = r alsó háromszögű rendszert előretolással oldjuk meg.
    """
    x = x0.copy().astype(float)
    r = b - A @ x
    n = len(b)
    L, D, U = split_LDU(A)
    LD = L + D  # alsó háromszög (főátlóval)
    res_history = [np.linalg.norm(r, ord=np.inf)]

    for _ in range(max_iter):
        # (D+L)s = r  =>  előretolás
        s = np.zeros(n)
        for i in range(n):
            s[i] = (r[i] - LD[i, :i] @ s[:i]) / LD[i, i]

        x = x + s
        r = r - A @ s
        res_history.append(np.linalg.norm(r, ord=np.inf))
        if res_history[-1] < tol:
            break

    return x, res_history

In [ ]:
x_GS, res_GS = gauss_seidel(A_J, b_J, x0)
print(f'Gauss–Seidel megoldás: {x_GS}  ({len(res_GS)-1} iteráció)')
print(f'Jacobi iteráció:       {len(res_J)-1} iteráció')

# Spektrálsugarak
L_J2, D_J2, U_J2 = split_LDU(A_J)
LD_J = L_J2 + D_J2
B_S = -np.linalg.solve(LD_J, U_J2)
rho_S = np.max(np.abs(np.linalg.eigvals(B_S)))
rho_Jac = np.max(np.abs(np.linalg.eigvals(B_J)))
print(f'ρ(B_J) = {rho_Jac:.4f},  ρ(B_S) = {rho_S:.4f}')
print(f'ρ(B_J)² = {rho_Jac**2:.4f}  (tridiagonális: egyenlők-e? {np.isclose(rho_S, rho_Jac**2)})')

# Összehasonlítás
plt.figure(figsize=(8, 5))
plt.semilogy(res_J, 'b-o', markersize=4, label='Jacobi')
plt.semilogy(res_GS, 'r-s', markersize=4, label='Gauss–Seidel')
plt.xlabel('Iteráció (k)')
plt.ylabel('‖r^(k)‖∞')
plt.title('Jacobi vs. Gauss–Seidel konvergencia')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

---
## VIII. rész – Relaxált Gauss–Seidel iteráció $S(\omega)$

### Levezetés

A Gauss–Seidel lépéshez $\omega$ relaxációs paramétert adunk:

$$x_i^{(k+1)} = (1 - \omega)\, x_i^{(k)} + \omega \cdot x_{i,S}^{(k+1)},$$
ahol $x_{i,S}^{(k+1)}$ a hagyományos Seidel-módszer ($S(1)$) által adott:
$$x_{i,S}^{(k+1)} = \frac{-1}{a_{ii}}\left(\sum_{j=1}^{i-1} a_{ij} x_j^{(k+1)} + \sum_{j=i+1}^n a_{ij} x_j^{(k)} - b_i\right)$$

**Mátrixos definíció:**
$$x^{(k+1)} = \underbrace{(D+\omega L)^{-1}\left[(1-\omega)D - \omega U\right]}_{B_{S(\omega)}} x^{(k)} + (D+\omega L)^{-1}\omega b$$

**Algoritmus:**
$$(D + \omega L) s^{(k)} = \omega r^{(k)} \quad (\text{LER mo.}), \qquad x^{(k+1)} := x^{(k)} + s^{(k)}, \qquad r^{(k+1)} := r^{(k)} - As^{(k)}$$

### Konvergencia

**Tétel (szükséges feltétel):** Ha egy mátrixra az $S(\omega)$ módszer konvergens, akkor $0 < \omega < 2$.

**Tétel (szimm. pozdef. eset):** Ha $A$ szimmetrikus, pozitív definit és $\omega \in (0, 2)$, akkor $S(\omega)$ konvergens.

**Tétel (tridiagonális eset):** Ha $A$ tridiagonális, szimm. és pozitív definit, akkor:
- $\varrho(B_S) = \varrho(B_J)^2$
- Az optimális paraméter: $\displaystyle\omega_0 = \frac{2}{1 + \sqrt{1 - \varrho(B_J)^2}}$
- $\varrho(B_{S(\omega_0)}) = \omega_0 - 1 < \varrho(B_S) = \varrho(B_J)^2$

In [ ]:
def relaxed_gauss_seidel(A, b, x0, omega=1.0, tol=1e-8, max_iter=2000):
    """Relaxált Gauss–Seidel iteráció S(omega)."""
    x = x0.copy().astype(float)
    r = b - A @ x
    n = len(b)
    L, D, _ = split_LDU(A)
    DwL = D + omega * L  # alsó háromszög: D + omega*L
    res_history = [np.linalg.norm(r, ord=np.inf)]

    for _ in range(max_iter):
        # (D + omega*L) s = omega * r  =>  előretolás
        rhs = omega * r
        s = np.zeros(n)
        for i in range(n):
            s[i] = (rhs[i] - DwL[i, :i] @ s[:i]) / DwL[i, i]

        x = x + s
        r = r - A @ s
        res_history.append(np.linalg.norm(r, ord=np.inf))
        if res_history[-1] < tol:
            break

    return x, res_history

In [ ]:
# Optimális omega kiszámítása
rho_BJ = np.max(np.abs(np.linalg.eigvals(B_J)))
omega_opt = 2 / (1 + np.sqrt(1 - rho_BJ**2))
print(f'ρ(B_J) = {rho_BJ:.4f}')
print(f'Optimális ω₀ = {omega_opt:.4f}')

# omega sweep: iterációk száma ω függvényében
omega_vals = np.linspace(0.1, 1.95, 80)
iter_counts = []
for om in omega_vals:
    _, res = relaxed_gauss_seidel(A_J, b_J, x0, omega=om, tol=1e-8, max_iter=500)
    iter_counts.append(len(res) - 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(omega_vals, iter_counts, 'b-', linewidth=2)
axes[0].axvline(omega_opt, color='r', linestyle='--', label=f'ω₀ = {omega_opt:.3f}')
axes[0].set_xlabel('ω')
axes[0].set_ylabel('Iterációk száma')
axes[0].set_title('Relaxált Gauss–Seidel: iterációszám ω függvényében')
axes[0].legend()
axes[0].grid(True)

# Konvergencia összehasonlítás
_, res_S1    = relaxed_gauss_seidel(A_J, b_J, x0, omega=1.0, tol=1e-10, max_iter=500)
_, res_Sopt  = relaxed_gauss_seidel(A_J, b_J, x0, omega=omega_opt, tol=1e-10, max_iter=500)
axes[1].semilogy(res_J[:80],   'b-',  label='Jacobi J(1)')
axes[1].semilogy(res_S1[:80],  'g-',  label='Gauss–Seidel S(1)')
axes[1].semilogy(res_Sopt[:80],'r-',  label=f'Relaxált S(ω₀={omega_opt:.3f})')
axes[1].set_xlabel('Iteráció')
axes[1].set_ylabel('‖r^(k)‖∞')
axes[1].set_title('Konvergencia összehasonlítás')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f'Jacobi: {len(res_J)-1} it., Gauss–Seidel: {len(res_S1)-1} it., Relaxált GS (ω₀): {len(res_Sopt)-1} it.')

---
## IX. rész – Richardson-iteráció $R(p)$

Legyen $A \in \mathbb{R}^{n\times n}$ szimmetrikus, pozitív definit mátrix és $p \in \mathbb{R}$.

$$Ax = b \implies p \cdot Ax = pb \implies 0 = -pAx + pb \implies x = (I - pA)x + pb$$

**Definíció ($R(p)$):**
$$x^{(k+1)} = \underbrace{(I - pA)}_{B_{R(p)}} x^{(k)} + pb$$

**Algoritmus (reziduum-vektoros alak):**
$$s^{(k)} := p\, r^{(k)}, \quad x^{(k+1)} := x^{(k)} + s^{(k)}, \quad r^{(k+1)} := r^{(k)} - A s^{(k)}$$

**Megjegyzés:** Ha $D = \mathrm{diag}(a_{11}, \ldots, a_{nn})$ és a Richardson-iterációt a $D^{-1}Ax = D^{-1}b$ LER-re alkalmazzuk $p = 1$ esetén, az eredeti LER-re felírt $J(p)$ csillapított Jacobi-iterációt kapjuk.

### Konvergencia

**Tétel:** Ha $A$ szimm., pozdef., sajátértékei $m = \lambda_1 \le \cdots \le \lambda_n = M$, akkor $R(p)$ pontosan akkor konvergens minden kezdővektorra, ha
$$p \in \left(0, \frac{2}{M}\right).$$

**Optimális paraméter:**
$$p_0 = \frac{2}{M + m}, \qquad \varrho(B_{R(p_0)}) = \frac{M - m}{M + m} =: q$$

($A$ szimmmetriájából $B_{R(p)}$ is szimmetrikus, így spektrálsugara és kettes normája megegyezik: $q = \|B_{R(p_0)}\|_2$.)

In [ ]:
def richardson(A, b, x0, p, tol=1e-8, max_iter=1000):
    """Richardson-iteráció R(p) paraméterrel."""
    x = x0.copy().astype(float)
    r = b - A @ x
    res_history = [np.linalg.norm(r, ord=np.inf)]

    for _ in range(max_iter):
        s = p * r
        x = x + s
        r = r - A @ s
        res_history.append(np.linalg.norm(r, ord=np.inf))
        if res_history[-1] < tol:
            break

    return x, res_history

In [ ]:
# Kidolgozott példa (ea_08 alapján)
# A = [[3,-1],[-1,3]], b = [1,5], sajátértékek: 2 és 4
A_R = np.array([[3.0, -1.0], [-1.0, 3.0]])
b_R = np.array([1.0, 5.0])
x0_R = np.zeros(2)

eigvals = np.linalg.eigvalsh(A_R)
m_R, M_R = eigvals.min(), eigvals.max()
p0_R = 2 / (M_R + m_R)
q_R  = (M_R - m_R) / (M_R + m_R)

print(f'Sajátértékek: m={m_R:.0f}, M={M_R:.0f}')
print(f'Konvergens p tartomány: (0, {2/M_R:.4f})')
print(f'Optimális p₀ = {p0_R:.4f},  q = {q_R:.4f}')

# Különböző p értékek
p_vals = [0.1, 0.2, p0_R, 0.4, 0.49]
colors_R = ['purple', 'green', 'red', 'orange', 'brown']

plt.figure(figsize=(9, 5))
for pv, col in zip(p_vals, colors_R):
    x_r, res_r = richardson(A_R, b_R, x0_R, p=pv, tol=1e-10, max_iter=300)
    label = f'p = {pv:.3f}' + (' ← optimális' if np.isclose(pv, p0_R) else '')
    plt.semilogy(res_r, color=col, label=label)

plt.xlabel('Iteráció (k)')
plt.ylabel('‖r^(k)‖∞')
plt.title('Richardson-iteráció különböző p értékekre')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

x_ref_R = np.linalg.solve(A_R, b_R)
x_rich, _ = richardson(A_R, b_R, x0_R, p=p0_R)
print(f'Megoldás (Richardson): {x_rich}')
print(f'Megoldás (pontos):     {x_ref_R}')

---
## Összefoglalás

| Módszer | Átmenet mátrix $B$ | Konvergencia feltétele |
|---|---|---|
| Jacobi $J(1)$ | $-D^{-1}(L+U)$ | s.d.d. $\Rightarrow$ konvergens; $\varrho(B_J)<1$ ekvivalens |
| Csillapított Jacobi $J(\omega)$ | $(1-\omega)I - \omega D^{-1}(L+U)$ | $J(1)$ konvergens + $0<\omega<1$ |
| Gauss–Seidel $S(1)$ | $-(L+D)^{-1}U$ | s.d.d. vagy szimm.+pozdef. |
| Relaxált GS $S(\omega)$ | $(D+\omega L)^{-1}[(1-\omega)D-\omega U]$ | szükséges: $0<\omega<2$; szimm.+pozdef.: elégséges |
| Richardson $R(p)$ | $(I-pA)$ | szimm.+pozdef.: $p\in(0,2/M)$ |

**Tridiagonális szimm.+pozdef. esetén** optimális paraméterek:
$$\omega_0 = \frac{2}{1+\sqrt{1-\varrho(B_J)^2}}, \qquad p_0 = \frac{2}{M+m}$$